<div style="text-align: justify; max-width: 90ch">

# Lecture 01f: pydantic schemas and hosted inference

**Status: Optional / career-track.**

Two things production code does that the mandatory notebooks skipped. First, the JSON schema dictionaries of `lec_01c` and `lec_01d` are usually written as **pydantic models**,
Python classes that generate the schema and validate the answer. Second, the "one line swap" from `lec_01d` section 5: the same chain against a **hosted** model through the OpenAI-compatible API that almost every provider speaks,
including Ollama itself.

**What this notebook covers**

- A pydantic model as a typed description of an answer, and `with_structured_output` returning objects.
- The OpenAI-compatible API: one client, many providers, Ollama included.
- Swapping the chat model in a LangChain chain, and what changes (nothing) and what does not (cost, privacy, latency).

</div>

In [1]:
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal

In [2]:
# --- Course configuration: edit TIER only -------------------------------------
TIER = "cpu"  # 8-16 GB RAM, no GPU (default)
# TIER = "gpu"  # >= 8 GB VRAM: bigger, better answers (pull qwen3:8b first)

import os
from pathlib import Path

TIER = os.environ.get("COURSE_TIER", TIER)  # tools/run_all.py can override
os.environ.setdefault(
    "HF_HUB_DISABLE_PROGRESS_BARS", "1"
)  # no download bars in saved outputs
CHAT_MODEL = {"cpu": "qwen3:1.7b", "gpu": "qwen3:8b"}[TIER]
EMBED_MODEL = "nomic-embed-text"
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5010")
EXPERIMENT = "llm-course-01-basics"
REPO = Path.cwd().resolve().parents[1]  # this notebook runs from reading_material/
DATA_DIR = REPO / "data"  # gitignored runtime state (indexes, caches)

from llm_course.checks import check_ollama

check_ollama(OLLAMA_HOST, [CHAT_MODEL])

Ollama OK at http://localhost:11434; models ready: qwen3:1.7b


<div style="text-align: justify; max-width: 90ch">

## 1. A schema as a class

A **class** is a template for objects: it lists the fields an object has and their types. You have used classes all along (`DataFrame`, `Client`);
here you write one. `pydantic.BaseModel` turns the class into a validator: it produces the JSON schema for the model and checks the answer against it.

</div>

In [3]:
class ForumTag(BaseModel):
    """How a forum question is filed."""

    category: Literal["pandas", "plots", "sklearn", "python-basics", "other"]
    needs_code: bool = Field(description="whether the answer needs a code example")
    summary: str = Field(description="the question in at most ten words")


ForumTag.model_json_schema()

{'description': 'How a forum question is filed.',
 'properties': {'category': {'enum': ['pandas',
    'plots',
    'sklearn',
    'python-basics',
    'other'],
   'title': 'Category',
   'type': 'string'},
  'needs_code': {'description': 'whether the answer needs a code example',
   'title': 'Needs Code',
   'type': 'boolean'},
  'summary': {'description': 'the question in at most ten words',
   'title': 'Summary',
   'type': 'string'}},
 'required': ['category', 'needs_code', 'summary'],
 'title': 'ForumTag',
 'type': 'object'}

<div style="text-align: justify; max-width: 90ch">

That is the dictionary you typed by hand in `lec_01c`, generated from three annotated lines. `Literal[...]` became the `enum`;
a description alone would only be a hint, and a small model happily answers "Data Manipulation" to a hint. LangChain accepts the class directly and gives back an object, not a dict.

</div>

In [4]:
llm = ChatOllama(
    model=CHAT_MODEL,
    base_url=OLLAMA_HOST,
    reasoning=False,
    temperature=0,
    num_predict=150,
)

tag_prompt = ChatPromptTemplate.from_template(
    "Tag this Python-course forum question.\n\nQuestion: {question}"
)
tag_chain = tag_prompt | llm.with_structured_output(ForumTag, method="json_schema")

tag = tag_chain.invoke({"question": "How do I merge two DataFrames on a customer id?"})
print(type(tag).__name__, "->", tag.category, "|", tag.needs_code, "|", tag.summary)

ForumTag -> pandas | True | Merge two DataFrames on a customer ID using pandas


<div style="text-align: justify; max-width: 90ch">

`tag.category` instead of `tag["category"]`; a typo in the field name is an error in your editor, not a `KeyError` at runtime.

</div>

<div style="text-align: justify; max-width: 90ch">

## 2. The OpenAI-compatible API

OpenAI published the first widely used chat API, and every serious provider now accepts requests in the same format, Ollama included:
`http://localhost:11434/v1`. So the `openai` client is a universal remote: point `base_url` at a provider, pass its key, done.
Against Ollama the key is any non-empty string.

</div>

In [5]:
local = OpenAI(base_url=f"{OLLAMA_HOST}/v1", api_key="ollama")

completion = local.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {"role": "user", "content": "In one sentence, what is a DataFrame? /no_think"}
    ],
    temperature=0,
    max_tokens=80,
    reasoning_effort="none",  # Ollama maps this to think=False
)
print(completion.choices[0].message.content)
print(completion.usage)

A DataFrame is a two-dimensional data structure in Python, commonly used in pandas, that stores data in rows and columns, allowing for efficient manipulation and analysis of data.
CompletionUsage(completion_tokens=34, prompt_tokens=29, total_tokens=63, completion_tokens_details=None, prompt_tokens_details=None)


<div style="text-align: justify; max-width: 90ch">

The same format, a different `base_url`, and a key from a provider account, is a hosted model. The cell below runs only if you put a provider's endpoint and key in `.env`:

```text
HOSTED_BASE_URL=https://api.example.com/v1
HOSTED_API_KEY=...
HOSTED_MODEL=...
```

Any provider with an OpenAI-compatible endpoint works; the free tier of Hugging Face's inference providers (`https://router.huggingface.co/v1` with your `HF_TOKEN` as the key) is one option.

</div>

In [6]:
load_dotenv(REPO / ".env")

HOSTED_BASE_URL = os.environ.get("HOSTED_BASE_URL")
HOSTED_API_KEY = os.environ.get("HOSTED_API_KEY")
HOSTED_MODEL = os.environ.get("HOSTED_MODEL")

if HOSTED_BASE_URL and HOSTED_API_KEY and HOSTED_MODEL:
    hosted = OpenAI(base_url=HOSTED_BASE_URL, api_key=HOSTED_API_KEY)
    completion = hosted.chat.completions.create(
        model=HOSTED_MODEL,
        messages=[{"role": "user", "content": "In one sentence, what is a DataFrame?"}],
        temperature=0,
        max_tokens=80,
    )
    print(completion.choices[0].message.content)
    print(completion.usage)
else:
    print("No hosted endpoint configured in .env; skipping (this is fine).")

No hosted endpoint configured in .env; skipping (this is fine).


<div style="text-align: justify; max-width: 90ch">

## 3. The swap inside a chain

`ChatOpenAI` is LangChain's chat model for any OpenAI-compatible endpoint. Everything after the first line of the chain is untouched, and `mlflow.langchain.autolog()` would trace it the same way.

</div>

In [7]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a teaching assistant for a Python course. Answer in one sentence.",
        ),
        ("human", "{question}"),
    ]
)

if HOSTED_BASE_URL and HOSTED_API_KEY and HOSTED_MODEL:
    swapped_llm = ChatOpenAI(
        model=HOSTED_MODEL,
        base_url=HOSTED_BASE_URL,
        api_key=HOSTED_API_KEY,
        temperature=0,
    )
    print("using the hosted model")
else:
    swapped_llm = ChatOpenAI(
        model=CHAT_MODEL,
        base_url=f"{OLLAMA_HOST}/v1",
        api_key="ollama",
        temperature=0,
        reasoning_effort="none",
    )
    print("no hosted endpoint; using Ollama through its OpenAI-compatible API")

chain = prompt | swapped_llm | StrOutputParser()
print(chain.invoke({"question": "What does random_state do?"}))

no hosted endpoint; using Ollama through its OpenAI-compatible API


The `random_state` parameter in scikit-learn is used to set the random seed for the random number generator, ensuring reproducibility of results.


<div style="text-align: justify; max-width: 90ch">

### ⏸ Pause point

The code did not change; the requirements table did. Against a hosted model, re-read the rows for **privacy** (the question left the laptop),
**cost** (`usage` is now a bill), **latency** (network, but a far bigger machine), and **maintenance** (the provider can retire the model).
A system that can flip between the two with one line is a system whose requirements can change without a rewrite.

</div>

<div style="text-align: justify; max-width: 90ch">

## Recap

- A pydantic `BaseModel` is a typed schema: it generates the JSON schema and validates the answer into an object.
- The OpenAI-compatible API is the common format; Ollama serves it at `/v1`, so the `openai` client and `ChatOpenAI` work locally and hosted alike.
- Swapping the model is one line; what changes is the requirements table, not the chain.

Back to the mandatory path: **lecture 2**.

</div>